# Platinum 4 — catalog, search, gpt-4.1

Card and/or question → BPM-015 citations, then a gpt-4.1 procedure note. **2024 test is sealed.**

In [1]:
from __future__ import annotations

import json
import sys
from pathlib import Path

import pandas as pd
from IPython.display import Markdown, display

HERE = Path.cwd().resolve()
REPO = None
for p in [HERE, *HERE.parents]:
    if (p / "platinum4" / "results").exists() and (p / "src" / "gold").exists():
        REPO = p
        break
if REPO is None:
    raise FileNotFoundError("repo root not found")
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))

P3 = REPO / "platinum3" / "results"
P4 = REPO / "platinum4" / "results"
SAMPLES = P3 / "sample_cards.json"
PACKETS = P4 / "sample_bot_packets.json"
ANSWERS = P4 / "answers"
POLICY_INDEX = REPO / "docs" / "miso_policy" / "index"

samples = json.loads(SAMPLES.read_text(encoding="utf-8")) if SAMPLES.exists() else []
packets = json.loads(PACKETS.read_text(encoding="utf-8")) if PACKETS.exists() else []

display(Markdown("# Platinum 4 — catalog, search, gpt-4.1"))
display(Markdown(
    "Not a contest model. Planner tags decide which BPM-015 r33 units to read. "
    "**gpt-4.1** writes the note from those units. **2024 test is sealed.**"
))

# Platinum 4 — catalog, search, gpt-4.1

Not a contest model. Planner tags decide which BPM-015 r33 units to read. **gpt-4.1** writes the note from those units. **2024 test is sealed.**

## Catalog

In [2]:
display(Markdown("## Search catalog"))
cat_path = POLICY_INDEX / "search_catalog.json"
if not cat_path.exists():
    display(Markdown("_Missing search_catalog.json. Run `python scripts/build_miso_policy_index.py`._"))
else:
    cat = json.loads(cat_path.read_text(encoding="utf-8"))
    display(Markdown(
        f"Doc `{cat.get('doc_id')}` status **{cat.get('status')}**, "
        f"{cat.get('n_units')} units, effective {cat.get('effective_date')}."
    ))
    holes = cat.get("holes") or {}
    display(Markdown(
        f"Holes: stub risks `{', '.join(holes.get('stub_risks') or [])}`. "
        f"Not retrieved: `{', '.join(holes.get('not_retrieved') or [])}`. "
        f"GIQ coarser than DPP 1/2/3: `{holes.get('giq_coarser_than_dpp')}`."
    ))
    shelf_rows = []
    for sh in cat.get("shelves") or []:
        shelf_rows.append({
            "facet": sh.get("facet"),
            "tag": sh.get("tag"),
            "n": sh.get("n"),
            "example": ", ".join(sh.get("example_cites") or [])[:80],
        })
    if shelf_rows:
        counts = pd.DataFrame(shelf_rows).groupby("facet", as_index=False)["n"].sum()
        display(Markdown("### Shelves (unit counts by facet)"))
        display(counts)
    examples = cat.get("query_examples") or []
    if examples:
        display(Markdown("### Canned query routes"))
        display(pd.DataFrame(examples))

## Search catalog

Doc `bpm-015-r33-clean` status **current**, 170 units, effective 2026-07-01.

Holes: stub risks `developer_serial_quit, hazard_exposure, policy_incentive`. Not retrieved: `bpm-015-r32-clean, bpm-015-r33-redlines`. GIQ coarser than DPP 1/2/3: `True`.

### Shelves (unit counts by facet)

,facet,n
0,actor,506
1,claim_class,205
2,gi_phase,219
3,milestone,65
4,risk_code,20
5,topic,287


### Canned query routes

,question,expected_unit_ids
0,Who funds a restudy if a peer withdraws?,[bpm015-r33::5.4.6]
1,What happens at Decision Point II?,[bpm015-r33::5.3.3]
2,What delay clauses apply after GIA when COD ha...,"[bpm015-r33::7.3, bpm015-r33::7.7]"
3,What is the DPP Study Funding Deposit D2?,[bpm015-r33::4.2.4.5]


## Ontology graph

In [3]:
display(Markdown("## Ontology graph"))
gpath = POLICY_INDEX / "ontology_graph.json"
if not gpath.exists():
    display(Markdown("_Missing ontology_graph.json._"))
else:
    graph = json.loads(gpath.read_text(encoding="utf-8"))
    nodes = graph.get("nodes") or []
    edges = graph.get("edges") or []
    display(Markdown(f"{graph.get('n_nodes')} nodes, {graph.get('n_edges')} edges."))
    kinds = pd.Series([n.get("kind") for n in nodes]).value_counts().rename_axis("kind").reset_index(name="n")
    display(kinds)
    rels = pd.Series([e.get("rel") for e in edges]).value_counts().rename_axis("rel").reset_index(name="n")
    display(rels)
    seed_rows = []
    for e in edges:
        if e.get("rel") == "seed_for":
            seed_rows.append({"unit": e.get("from"), "risk": e.get("to")})
    if seed_rows:
        display(Markdown("### Seed edges (risk map, not every tagged descendant)"))
        display(pd.DataFrame(seed_rows))

## Ontology graph

219 nodes, 1617 edges.

,kind,n
0,unit,170
1,tag,39
2,risk,9
3,doc,1


,rel,n
0,tagged_with,1282
1,contains,170
2,parent_of,145
3,seed_for,20


### Seed edges (risk map, not every tagged descendant)

,unit,risk
0,unit:bpm015-r33::5.2.5,risk:abandonment
1,unit:bpm015-r33::5.3.5,risk:abandonment
2,unit:bpm015-r33::4.2.4.6,risk:abandonment
3,unit:bpm015-r33::6.2.11,risk:abandonment
4,unit:bpm015-r33::5.2.3,risk:abandonment
5,unit:bpm015-r33::5.3.3,risk:abandonment
6,unit:bpm015-r33::3.1.1,risk:system_congestion
7,unit:bpm015-r33::4.3,risk:system_congestion
8,unit:bpm015-r33::6.2.7,risk:gia_execution
9,unit:bpm015-r33::6.2.8,risk:gia_execution


## Contract

In [4]:
display(Markdown("## What GPT is allowed to see"))
display(Markdown(
    "Input is a Platinum 3 risk card and/or a question. Retrieval is from "
    "`docs/miso_policy/index/` (r33 clean). Compliance is **not_determined**. "
    "Model is **gpt-4.1** only. Do not send `val_card_audit.parquet`."
))
ont_path = POLICY_INDEX / "ontology.json"
if ont_path.exists():
    ont = json.loads(ont_path.read_text(encoding="utf-8"))
    display(Markdown(
        f"Ontology `{ont.get('schema_version')}`. "
        f"Stub risks (no BPM clause): `{', '.join(ont.get('stub_risks_no_bpm') or [])}`."
    ))
risk_map_path = POLICY_INDEX / "maps" / "risk_code_to_units.json"
if risk_map_path.exists():
    risk_map = json.loads(risk_map_path.read_text(encoding="utf-8"))
    map_rows = []
    for code, entry in risk_map.items():
        map_rows.append({
            "risk_code": code,
            "n_units": len(entry.get("unit_ids") or []),
            "note": entry.get("note") or "BPM-015 clause(s)",
            "unit_ids": ", ".join(entry.get("unit_ids") or []) or "—",
        })
    display(pd.DataFrame(map_rows))

## What GPT is allowed to see

Input is a Platinum 3 risk card and/or a question. Retrieval is from `docs/miso_policy/index/` (r33 clean). Compliance is **not_determined**. Model is **gpt-4.1** only. Do not send `val_card_audit.parquet`.

Ontology `policy_index.v1`. Stub risks (no BPM clause): `developer_serial_quit, hazard_exposure, policy_incentive`.

,risk_code,n_units,note,unit_ids
0,abandonment,6,BPM-015 clause(s),"bpm015-r33::5.2.5, bpm015-r33::5.3.5, bpm015-r..."
1,system_congestion,2,BPM-015 clause(s),"bpm015-r33::3.1.1, bpm015-r33::4.3"
2,gia_execution,5,BPM-015 clause(s),"bpm015-r33::6.2.7, bpm015-r33::6.2.8, bpm015-r..."
3,cod_already_slipped,3,BPM-015 clause(s),"bpm015-r33::4.2, bpm015-r33::7.3, bpm015-r33::7.7"
4,developer_serial_quit,0,No BPM-015 clause. Keep company/stub playbook ...,—
5,restudy_friction,1,BPM-015 clause(s),bpm015-r33::5.4.6
6,hazard_exposure,0,No BPM-015 clause. Keep company/stub playbook ...,—
7,cost_pressure,3,BPM-015 clause(s),"bpm015-r33::4.2.4.5, bpm015-r33::4.2.4, bpm015..."
8,policy_incentive,0,No BPM-015 clause. Keep company/stub playbook ...,—


## Card-only retrieval

In [5]:
from src.modeling.platinum4.retrieve import retrieve_for_card

if not packets and samples:
    try:
        packets = [retrieve_for_card(c) for c in samples if (c.get("scenario") or {}).get("name") == "baseline"]
    except FileNotFoundError as exc:
        display(Markdown(f"_Index missing: {exc}_"))
        packets = []

display(Markdown("## Card-only retrieval (`retrieve_for_card`)"))
if packets:
    shown = 0
    for pkt in packets:
        card = pkt.get("card") or {}
        if (card.get("scenario") or {}).get("name") not in {"baseline", "restudy"}:
            continue
        if shown >= 6:
            break
        shown += 1
        cites = "\n".join(
            f"- `{r.get('cite')}` — {r.get('title')}" for r in (pkt.get("retrieved") or [])
        ) or "- _none_"
        stakeholders = ", ".join(
            f"`{s.get('actor')}`" for s in (pkt.get("stakeholders") or [])
        ) or "_none_"
        display(Markdown(
            f"### `{card.get('project_key')}` · {(card.get('scenario') or {}).get('name')} · {card.get('study_phase')}\n"
            f"GI phase guess: `{pkt.get('workflow', {}).get('gi_phase_guess')}`\n\n"
            f"Compliance: **{(pkt.get('compliance') or {}).get('status')}** — "
            f"{(pkt.get('compliance') or {}).get('meaning')}\n\n"
            f"Stakeholders: {stakeholders}\n\n"
            f"{cites}"
        ))
else:
    display(Markdown(
        "_No bot packets. Run `python scripts/build_miso_policy_index.py` "
        "(needs Platinum 3 `sample_cards.json`)._"
    ))

## Card-only retrieval (`retrieve_for_card`)

### `P::B::E291` · baseline · IA Executed
GI phase guess: `['gia', 'post_gia']`

Compliance: **not_determined** — We do not score legal compliance of a COD slip. Retrieved delay/restudy/withdrawal clauses only.

Stakeholders: `ic`, `miso`, `to`, `affected_system`, `ferc`

- `BPM-015 r33 §7.3` — Interconnection Customer delays
- `BPM-015 r33 §7.7` — Commercial Operation
- `BPM-015 r33 §4.2` — Initial Screening
- `BPM-015 r33 §6.2.7` — Submittal of IA for Appendix Review
- `BPM-015 r33 §6.2.8` — Submittal of GIA/FCA for Execution / Filing Unexecuted
- `BPM-015 r33 §6.2.9` — Provisional Generator Interconnection Agreement
- `BPM-015 r33 §4.3` — Determination of Project Linkages and Potential Grouping
- `BPM-015 r33 §4.2.4.6` — Refunds of Study Deposits
- `BPM-015 r33 §3.1.1` — Contour Map
- `BPM-015 r33 §5.2.3` — Interconnection Customer Decision Point I
- `BPM-015 r33 §5.2.5` — Withdrawal from DPP Phase I
- `BPM-015 r33 §5.3.3` — Interconnection Customer Decision Point II

### `P::B::E291` · restudy · IA Executed
GI phase guess: `['gia', 'post_gia']`

Compliance: **not_determined** — We do not score legal compliance of a COD slip. Retrieved delay/restudy/withdrawal clauses only.

Stakeholders: `ic`, `miso`, `to`, `affected_system`, `ferc`

- `BPM-015 r33 §7.3` — Interconnection Customer delays
- `BPM-015 r33 §7.7` — Commercial Operation
- `BPM-015 r33 §4.2` — Initial Screening
- `BPM-015 r33 §4.2.4.6` — Refunds of Study Deposits
- `BPM-015 r33 §5.4.6` — Interconnection Study Restudy
- `BPM-015 r33 §6.2.7` — Submittal of IA for Appendix Review
- `BPM-015 r33 §6.2.8` — Submittal of GIA/FCA for Execution / Filing Unexecuted
- `BPM-015 r33 §6.2.9` — Provisional Generator Interconnection Agreement
- `BPM-015 r33 §7.1` — Suspension
- `BPM-015 r33 §4.3` — Determination of Project Linkages and Potential Grouping
- `BPM-015 r33 §3.1.1` — Contour Map
- `BPM-015 r33 §5.2.3` — Interconnection Customer Decision Point I

### `P::J2280` · baseline · System Impact Study
GI phase guess: `['dpp1', 'dpp2']`

Compliance: **not_determined** — We do not score legal compliance of a COD slip. Retrieved delay/restudy/withdrawal clauses only.

Stakeholders: `ic`, `miso`, `to`, `affected_system`

- `BPM-015 r33 §4.2` — Initial Screening
- `BPM-015 r33 §7.3` — Interconnection Customer delays
- `BPM-015 r33 §7.7` — Commercial Operation
- `BPM-015 r33 §6.2.11` — Refunds of Definitive Planning Phase Milestones (M2, M3, M4)
- `BPM-015 r33 §4.2.4.6` — Refunds of Study Deposits
- `BPM-015 r33 §5.2.3` — Interconnection Customer Decision Point I
- `BPM-015 r33 §5.2.5` — Withdrawal from DPP Phase I
- `BPM-015 r33 §5.3.3` — Interconnection Customer Decision Point II
- `BPM-015 r33 §5.3.5` — Withdrawal from DPP Phase II
- `BPM-015 r33 §4.3` — Determination of Project Linkages and Potential Grouping
- `BPM-015 r33 §3.1.1` — Contour Map
- `BPM-015 r33 §5.1.2` — Site Control Requirements Review Detail

### `P::J2280` · restudy · System Impact Study
GI phase guess: `['dpp1', 'dpp2']`

Compliance: **not_determined** — We do not score legal compliance of a COD slip. Retrieved delay/restudy/withdrawal clauses only.

Stakeholders: `ic`, `miso`, `to`, `affected_system`, `ferc`

- `BPM-015 r33 §4.2.4.6` — Refunds of Study Deposits
- `BPM-015 r33 §4.2` — Initial Screening
- `BPM-015 r33 §7.3` — Interconnection Customer delays
- `BPM-015 r33 §7.7` — Commercial Operation
- `BPM-015 r33 §5.4.6` — Interconnection Study Restudy
- `BPM-015 r33 §6.2.11` — Refunds of Definitive Planning Phase Milestones (M2, M3, M4)
- `BPM-015 r33 §5.2.3` — Interconnection Customer Decision Point I
- `BPM-015 r33 §5.2.5` — Withdrawal from DPP Phase I
- `BPM-015 r33 §5.3.3` — Interconnection Customer Decision Point II
- `BPM-015 r33 §5.3.5` — Withdrawal from DPP Phase II
- `BPM-015 r33 §4.3` — Determination of Project Linkages and Potential Grouping
- `BPM-015 r33 §3.1.1` — Contour Map

### `P::J2460` · baseline · Not Started
GI phase guess: `['pre_queue']`

Compliance: **not_determined** — We do not score legal compliance of a COD slip. Retrieved delay/restudy/withdrawal clauses only.

Stakeholders: `ic`, `miso`, `to`, `affected_system`, `ferc`

- `BPM-015 r33 §3.1.1` — Contour Map
- `BPM-015 r33 §6.2.11` — Refunds of Definitive Planning Phase Milestones (M2, M3, M4)
- `BPM-015 r33 §4.3` — Determination of Project Linkages and Potential Grouping
- `BPM-015 r33 §4.2.4.6` — Refunds of Study Deposits
- `BPM-015 r33 §5.2.3` — Interconnection Customer Decision Point I
- `BPM-015 r33 §5.2.5` — Withdrawal from DPP Phase I
- `BPM-015 r33 §5.3.3` — Interconnection Customer Decision Point II
- `BPM-015 r33 §5.3.5` — Withdrawal from DPP Phase II
- `BPM-015 r33 §5.4.6` — Interconnection Study Restudy
- `BPM-015 r33 §4.2.2.1` — Requirements
- `BPM-015 r33 §5.1.2` — Site Control Requirements Review Detail
- `BPM-015 r33 §3.1.2` — Ongoing Efforts

### `P::J2460` · restudy · Not Started
GI phase guess: `['pre_queue']`

Compliance: **not_determined** — We do not score legal compliance of a COD slip. Retrieved delay/restudy/withdrawal clauses only.

Stakeholders: `ic`, `miso`, `to`, `ferc`, `affected_system`, `lse`

- `BPM-015 r33 §4.2.4.6` — Refunds of Study Deposits
- `BPM-015 r33 §5.4.6` — Interconnection Study Restudy
- `BPM-015 r33 §3.1.1` — Contour Map
- `BPM-015 r33 §6.2.11` — Refunds of Definitive Planning Phase Milestones (M2, M3, M4)
- `BPM-015 r33 §4.3` — Determination of Project Linkages and Potential Grouping
- `BPM-015 r33 §5.2.3` — Interconnection Customer Decision Point I
- `BPM-015 r33 §5.2.5` — Withdrawal from DPP Phase I
- `BPM-015 r33 §5.3.3` — Interconnection Customer Decision Point II
- `BPM-015 r33 §5.3.5` — Withdrawal from DPP Phase II
- `BPM-015 r33 §4.2.2.1` — Requirements
- `BPM-015 r33 §4.5.1` — Applicable Transmission Owner Planning Criteria - General
- `BPM-015 r33 §5.1.2` — Site Control Requirements Review Detail

## Hybrid search

In [6]:
from src.modeling.platinum4.errors import BotError
from src.modeling.platinum4.search import retrieve_for_query

display(Markdown("## Card + question (`retrieve_for_query`)"))
LOOK = "P::B::E291"
card = next((c for c in samples if c.get("project_key") == LOOK and (c.get("scenario") or {}).get("name") == "baseline"), None)
question = "Who funds a restudy if a peer withdraws?"
query_pkt = None
try:
    query_pkt = retrieve_for_query({
        "question": question,
        "card": card,
        "requirements": {
            "audience": "analyst",
            "dialect": "bpm_register",
            "need": ["workflow", "stakeholders", "citations", "do_not_claim"],
            "max_units": 12,
        },
    })
except BotError as exc:
    display(Markdown(f"_Query failed `{exc.code}`: {exc}_"))
except FileNotFoundError as exc:
    display(Markdown(f"_Index missing: {exc}_"))

if query_pkt:
    ids = [r.get("unit_id") for r in (query_pkt.get("retrieved") or [])]
    display(Markdown(
        f"Question: **{question}**  \n"
        f"Kept units: `{', '.join(ids) or 'none'}`.  \n"
        f"Gaps: {query_pkt.get('gaps') or []}."
    ))
    trace = query_pkt.get("search_trace") or {}
    alg_rows = []
    for a in trace.get("algorithms") or []:
        alg_rows.append({
            "algorithm": a.get("name"),
            "n": a.get("n_candidates"),
            "top": ", ".join(a.get("top_ids") or [])[:80],
            "ms": a.get("elapsed_ms"),
            "error": a.get("error"),
        })
    display(Markdown("### search_trace.algorithms"))
    display(pd.DataFrame(alg_rows))
    dir_rows = []
    for d in trace.get("directives") or []:
        dir_rows.append({"directive": d.get("name"), "ok": d.get("ok"), "reason": d.get("reason")})
    display(Markdown("### search_trace.directives"))
    display(pd.DataFrame(dir_rows))
    gslice = query_pkt.get("graph_slice") or {}
    display(Markdown(
        f"graph_slice: {len(gslice.get('nodes') or [])} nodes, "
        f"{len(gslice.get('edges') or [])} edges, "
        f"algorithms `{', '.join(gslice.get('algorithms') or [])}`."
    ))

## Card + question (`retrieve_for_query`)

Question: **Who funds a restudy if a peer withdraws?**  
Kept units: `bpm015-r33::5.4.6, bpm015-r33::7.3, bpm015-r33::7.7, bpm015-r33::4.2, bpm015-r33::4.2.4.6, bpm015-r33::7.1, bpm015-r33::6.2.8, bpm015-r33::6.2.7, bpm015-r33::6.2.11, bpm015-r33::4.3, bpm015-r33::5.2.3, bpm015-r33::5.2.5`.  
Gaps: ['developer_serial_quit has no BPM-015 clause'].

### search_trace.algorithms

,algorithm,n,top,ms,error
0,alg_catalog_route,128,"bpm015-r33::7.3, bpm015-r33::7.7, bpm015-r33::...",0,None
1,alg_lexicon,4,,0,None
2,alg_seed_lookup,15,"bpm015-r33::3.1.1, bpm015-r33::4.3, bpm015-r33...",1,None
3,alg_inverted_filter,106,"bpm015-r33::5.5, bpm015-r33::5.5.1, bpm015-r33...",0,None
4,alg_phrase_match,0,,1,None
5,alg_bm25,77,"bpm015-r33::5.4.6, bpm015-r33::4.2.4.6, bpm015...",13,None
6,alg_fusion,120,"bpm015-r33::5.4.6, bpm015-r33::7.3, bpm015-r33...",0,None
7,alg_coverage_pass,0,,0,None


### search_trace.directives

,directive,ok,reason
0,DIR_EMPTY_INPUT,True,question and/or card present
1,DIR_REQUIRE_CURRENT_DOC,True,kept status:current r33 units only
2,DIR_NO_R32,True,r32/redline unit ids never added
3,DIR_STUB_HONESTY,True,stub risks retrieved nothing; gaps recorded
4,DIR_DROP_LEGAL_OPINION,True,dropped do_not_use_as_legal_opinion
5,DIR_PREFER_WORKFLOW,True,study_method downranked unless the question is...
6,DIR_AUDIENCE_ACTOR,True,audience=analyst; no ic filter
7,DIR_FAMILY_CAP,True,"max 2 units per section family, 3 if seed"
8,DIR_COVERAGE,True,all lexicon topics/risks have a kept unit


graph_slice: 54 nodes, 161 edges, algorithms `alg_catalog_route, alg_lexicon, alg_seed_lookup, alg_inverted_filter, alg_phrase_match, alg_bm25, alg_fusion, alg_coverage_pass`.

## Composed note

In [7]:
from src.modeling.platinum4.compose import PINNED_MODEL, compose

display(Markdown("## Three-pass note (gpt-4.1)"))
display(Markdown(f"Pinned model: `{PINNED_MODEL}`. If `OPENAI_API_KEY` is missing, the skeleton still runs pass 4."))
answer_md = None
answer_meta = None
if ANSWERS.exists():
    mds = sorted(ANSWERS.glob("*.md"))
    prefer = [p for p in mds if "restudy" in p.name or "withdraw" in p.name]
    pick = (prefer or mds)
    if pick:
        answer_md = pick[0].read_text(encoding="utf-8")
        js = pick[0].with_suffix(".json")
        if js.exists():
            answer_meta = json.loads(js.read_text(encoding="utf-8"))

if answer_md is None and query_pkt is not None:
    composed = compose(query_pkt, dialect="bpm_register")
    answer_md = composed.get("markdown")
    answer_meta = composed

if answer_md:
    if answer_meta:
        passes = answer_meta.get("passes") or []
        display(Markdown(
            f"compose_mode `{answer_meta.get('compose_mode')}`, "
            f"model `{answer_meta.get('model') or PINNED_MODEL}`."
        ))
        if passes:
            display(pd.DataFrame(passes))
    display(Markdown(answer_md))
else:
    display(Markdown(
        "_No composed note. Run "
        "`python scripts/run_platinum4_answer.py --question \"Who funds a restudy if a peer withdraws?\" --project-key P::B::E291`._"
    ))

## Three-pass note (gpt-4.1)

Pinned model: `gpt-4.1`. If `OPENAI_API_KEY` is missing, the skeleton still runs pass 4.

compose_mode `gpt-4.1`, model `gpt-4.1`.

,name,ok,elapsed_ms,tokens,model,dirty,flags
0,pass1_draft,True,4460,"{'prompt_tokens': 7580, 'completion_tokens': 7...",gpt-4.1,NaN,NaN
1,pass2_critic,True,5703,"{'prompt_tokens': 7856, 'completion_tokens': 8...",gpt-4.1,True,NaN
2,pass3_repair,True,4627,"{'prompt_tokens': 10092, 'completion_tokens': ...",gpt-4.1,NaN,NaN
3,pass4_postcheck,True,1,NaN,gpt-4.1,NaN,[]


# Interconnection procedure note

## Situation
The inquiry concerns who funds a restudy if a peer interconnection request (IR) withdraws from the MISO queue. The project card is in the Generator Interconnection Agreement (GIA) or post-GIA phase as of the observation date. This note restates BPM-015 r33 procedure. It is not a legal opinion. It does not score compliance of a Commercial Operation Date (COD) slip.

## Workflow
If MISO determines that a restudy of an Interconnection Study is required because an IR withdraws or is deemed withdrawn before all Generator Interconnection Agreements (GIAs), Facilities Construction Agreements (FCAs), and/or Multi-Party Facilities Construction Agreements (MPFCAs) for that Definitive Planning Phase (DPP) cycle have been executed or filed unexecuted with FERC, MISO shall provide notice of a restudy as necessary (BPM-015 r33 §5.4.6). The notice includes a preliminary analysis, an explanation of why restudy is required, and a good faith estimate of the cost to perform the restudy. The Interconnection Customer (IC) shall notify MISO within five (5) Business Days whether it wishes to proceed with the restudy or withdraw its IR. If the IC does not respond, MISO will deem the IR withdrawn (BPM-015 r33 §5.4.6). The Interconnection Study restudy will be performed according to the Generator Interconnection Procedures (GIP) and the BPMs in effect at the time the notice is given by MISO.

The IC funds the restudy from any remaining study deposit or from an additional deposit as noticed by MISO (BPM-015 r33 §5.4.6). Refunds of study deposits and milestone payments are governed by the timing of withdrawal and the applicable DPP phase. If the IR is withdrawn after certain decision points, milestone payments may become non-refundable and are used to fund study costs and Network Upgrades (BPM-015 r33 §4.2.4.6; §6.2.11).

## Stakeholders
- The Interconnection Customer funds the restudy using remaining or additional study deposits as required by MISO (BPM-015 r33 §5.4.6).
- MISO (Transmission Provider) issues the restudy notice, provides cost estimates, and performs the restudy.
- The Transmission Owner coordinates on affected facilities and participates in the restudy process where applicable.

## Citations
- BPM-015 r33 §5.4.6 - Interconnection Study Restudy (pages 56-56)
- BPM-015 r33 §4.2.4.6 - Refunds of Study Deposits (pages 34-35)
- BPM-015 r33 §6.2.11 - Refunds of Definitive Planning Phase Milestones (M2, M3, M4) (pages 92-95)

## Gaps
- No BPM-015 clause was retrieved for developer serial quit.
- The GIP (Attachment X) is the tariff and is not in this packet.

## Must not claim
- do not quote months of future COD slip as a model output
- do not claim a MISO Step-Up / Firm Service failure record
- do not treat EIA delayed MW as a complete GIA list (overlay is thin)
- do not treat scenario deltas as causal effects
- do not use FERC-730 as a clean delay source
- do not unseal 2024 test labels or tune on test/score


Compliance: not_determined.


In [8]:
display(Markdown("## What this is not"))
display(Markdown(
    "- Not a new neural net.\n"
    "- Not months of future COD slip (Platinum 1 failed; do not quote it).\n"
    "- Not a legal opinion on a COD slip (`compliance` is `not_determined`).\n"
    "- `developer_serial_quit`, `hazard_exposure`, and `policy_incentive` have no BPM clause.\n"
    "- 2024 test labels stay sealed. Packets are 2023 val samples only.\n"
    "- No chat model other than gpt-4.1."
))
display(Markdown(
    "Rebuild cards: `python scripts/run_platinum3_risk_cards.py`  \n"
    "Rebuild index + packets: `python scripts/build_miso_policy_index.py`  \n"
    "Compose a note: `python scripts/run_platinum4_answer.py --question \"Who funds a restudy if a peer withdraws?\" --project-key P::B::E291`  \n"
    "Rebuild this notebook: `python scripts/build_platinum4_report.py`"
))

## What this is not

- Not a new neural net.
- Not months of future COD slip (Platinum 1 failed; do not quote it).
- Not a legal opinion on a COD slip (`compliance` is `not_determined`).
- `developer_serial_quit`, `hazard_exposure`, and `policy_incentive` have no BPM clause.
- 2024 test labels stay sealed. Packets are 2023 val samples only.
- No chat model other than gpt-4.1.

Rebuild cards: `python scripts/run_platinum3_risk_cards.py`  
Rebuild index + packets: `python scripts/build_miso_policy_index.py`  
Compose a note: `python scripts/run_platinum4_answer.py --question "Who funds a restudy if a peer withdraws?" --project-key P::B::E291`  
Rebuild this notebook: `python scripts/build_platinum4_report.py`